Tokenizing text

In [20]:
import torch # <---
from importlib.metadata import version

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

Device: cuda
torch version: 2.10.0+cu128
tiktoken version: 0.12.0


In [21]:
import os
import requests

# قائمة بكتب من Project Gutenberg لضمان تجاوز حجم 2MB
books = {
    "sherlock_holmes": "https://www.gutenberg.org/files/1661/1661-0.txt",
    "moby_dick": "https://www.gutenberg.org/files/2701/2701-0.txt",
    "great_expectations": "https://www.gutenberg.org/files/1400/1400-0.txt"
}

output_file = "pretraining_data.txt"

# التأكد من عدم تحميل الملفات مرتين
if not os.path.exists(output_file):
    with open(output_file, "w", encoding="utf-8") as outfile:
        for name, url in books.items():
            print(f"Downloading {name}...")
            response = requests.get(url, timeout=30)
            response.raise_for_status()

            # فك ترميز النص وحفظه
            text = response.text
            outfile.write(text + "\n\n")
            print(f"Added {name} to {output_file}")

# التحقق من الحجم
file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"Successfully created {output_file} with size: {file_size_mb:.2f} MB")

Added sherlock_holmes to pretraining_data.txt
Added moby_dick to pretraining_data.txt
Added great_expectations to pretraining_data.txt
Successfully created pretraining_data.txt with size: 2.75 MB


In [22]:
# نقرأ الملف الجديد الذي قمنا بتجميع الكتب فيه
with open("pretraining_data.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# الآن لنعرض النتائج
print("Total number of characters:", len(raw_text))
print("--- Sample of the text ---")
print(raw_text[:200]) # عرضت لك أول 200 حرف بدلاً من 99 لتأخذ فكرة أوضح عن البداية

Total number of characters: 2795192
--- Sample of the text ---
﻿The Project Gutenberg eBook of The Adventures of Sherlock Holmes,
by Arthur Conan Doyle

This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost


In [23]:
import re

text = "Hello, world. This, is a test."
# هذا النمط يفصل الكلمات وعلامات الترقيم
tokens = re.findall(r"\w+|[^\w\s]", text)

print(tokens)
# النتيجة ستكون: ['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [24]:
import re

text = "Hello, world. This, is a test."
# هذا النمط يفصل المسافات وأيضاً علامات الترقيم كرموز منفصلة
result = re.findall(r"\w+|[^\w\s]|\s+", text)

print(result)
# النتيجة: ['Hello', ',', ' ', 'world', '.', ' ', 'This', ',', ' ', 'is', ' ', 'a', ' ', 'test', '.']

['Hello', ',', ' ', 'world', '.', ' ', 'This', ',', ' ', 'is', ' ', 'a', ' ', 'test', '.']


In [25]:
result = re.split(r'([,.]|\s)', text)

print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [26]:
# Strip whitespace from each item and then filter out any empty strings.
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [27]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [28]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['\ufeffThe', 'Project', 'Gutenberg', 'eBook', 'of', 'The', 'Adventures', 'of', 'Sherlock', 'Holmes', ',', 'by', 'Arthur', 'Conan', 'Doyle', 'This', 'eBook', 'is', 'for', 'the', 'use', 'of', 'anyone', 'anywhere', 'in', 'the', 'United', 'States', 'and', 'most']


In [29]:
print(len(preprocessed))

596168


In [30]:
len(set(preprocessed)) # drop duplicates

31594

In [31]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

31594


In [32]:
for integer,token in enumerate(all_words):
    print(integer,token)

Streaming output truncated to the last 5000 lines.
26594 three-masted
26595 three-year
26596 three-years’
26597 threepence
26598 threes
26599 three—oh
26600 three—or
26601 threshold
26602 thresholds
26603 threw
26604 thrice
26605 thrill
26606 thrilled
26607 thrilling
26608 thrills
26609 thrive
26610 thriven
26611 thriving
26612 throat
26613 throats
26614 throb
26615 throbbed
26616 throbbing
26617 throbbings
26618 throbs
26619 throes
26620 throne
26621 throned
26622 thrones
26623 thronged
26624 throttle
26625 throttled
26626 through
26627 throughout
26628 through—I
26629 throw
26630 throwing
26631 thrown
26632 throws
26633 thro’
26634 thrush
26635 thrust
26636 thrusting
26637 thrusts
26638 thud
26639 thudding
26640 thumb
26641 thumb-end
26642 thumb-nails
26643 thumb-worn
26644 thumbs
26645 thump
26646 thumped
26647 thumping
26648 thunder
26649 thunder-boom
26650 thunder-claps
26651 thunder-clotted
26652 thunder-clouds
26653 thunder-cloven
26654 thunder-heads
26655 thunderbolts
26656 thu

In [33]:
vocab = {token:integer for integer,token in enumerate(all_words)}

In [34]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('#1661]', 1)
('$1', 2)
('$20', 3)
('$5', 4)
('$7', 5)
('&', 6)
('&c', 7)
('(', 8)
(')', 9)
('*', 10)
('***', 11)
('*A', 12)
('*Bible', 13)
('*I', 14)
('*In', 15)
('*It', 16)
('*Partly', 17)
('*Quoin', 18)
('*See', 19)
('*Since', 20)
('*That', 21)
('*The', 22)
('*This', 23)
('*Though', 24)
('*Why', 25)
('*With', 26)
('*—These', 27)
(',', 28)
('-', 29)
('--', 30)
('-a', 31)
('.', 32)
('000', 33)
('1', 34)
('10', 35)
('100', 36)
('1000', 37)
('101', 38)
('102', 39)
('103', 40)
('104', 41)
('105', 42)
('106', 43)
('107', 44)
('108', 45)
('109', 46)
('11', 47)
('110', 48)
('111', 49)
('112', 50)


In [35]:
{i:s for s,i in vocab.items()}

{0: '!',
 1: '#1661]',
 2: '$1',
 3: '$20',
 4: '$5',
 5: '$7',
 6: '&',
 7: '&c',
 8: '(',
 9: ')',
 10: '*',
 11: '***',
 12: '*A',
 13: '*Bible',
 14: '*I',
 15: '*In',
 16: '*It',
 17: '*Partly',
 18: '*Quoin',
 19: '*See',
 20: '*Since',
 21: '*That',
 22: '*The',
 23: '*This',
 24: '*Though',
 25: '*Why',
 26: '*With',
 27: '*—These',
 28: ',',
 29: '-',
 30: '--',
 31: '-a',
 32: '.',
 33: '000',
 34: '1',
 35: '10',
 36: '100',
 37: '1000',
 38: '101',
 39: '102',
 40: '103',
 41: '104',
 42: '105',
 43: '106',
 44: '107',
 45: '108',
 46: '109',
 47: '11',
 48: '110',
 49: '111',
 50: '112',
 51: '113',
 52: '114',
 53: '115',
 54: '116',
 55: '117',
 56: '118',
 57: '119',
 58: '12',
 59: '120',
 60: '121',
 61: '122',
 62: '123',
 63: '124',
 64: '125',
 65: '126',
 66: '127',
 67: '128',
 68: '129',
 69: '13',
 70: '130',
 71: '131',
 72: '132',
 73: '133',
 74: '134',
 75: '135',
 76: '14',
 77: '140',
 78: '1400',
 79: '144',
 80: '1492',
 81: '15',
 82: '150',
 83: '1500

In [36]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [37]:
" ".join(['i', 'love'])

'i love'

In [38]:
def encode(self, text):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]

    # الحل هنا: استخدام .get(s, 0) بدلاً من البحث المباشر في القاموس
    # هذا سيضع 0 لأي رمز غير موجود في القاموس بدلاً من إحداث KeyError
    ids = [self.str_to_int.get(s, 0) for s in preprocessed]
    return ids

In [39]:
tokenizer.encode("and established himself in a villa ")

[392, 4920, 2241, 287, 257, 4489, 64, 220]

In [40]:
tokenizer.decode([5847, 12203, 15193, 15872, 5132, 28330])

'iserARE Performance Soccerorial Holt'

In [41]:
def decode(self, ids):
    # نستخدم get مع قيمة افتراضية "<|unk|>" إذا كان الرقم غير موجود
    text = " ".join([self.int_to_str.get(i, "<|unk|>") for i in ids])

    # تحسين شكل النص (إزالة المسافات قبل علامات الترقيم)
    text = text.replace(" ,", ",").replace(" .", ".").replace(" ?", "?").replace(" !", "!")
    return text

In [42]:
# جرب تحويل قائمة بها رقم 0 (الذي قد يسبب المشكلة)
ids_test = [158, 398, 0, 569, 116]
print(tokenizer.decode(ids_test))

�rom! V�


In [43]:
import tiktoken

# 1. احصل على التوكنايزر بشكل طبيعي
tokenizer = tiktoken.get_encoding("gpt2")

# 2. جرب التشفير وفك التشفير مباشرة، سيعمل بدون مشاكل مع الرموز
text = "The quick brown fox's jump."
encoded = tokenizer.encode(text)
decoded = tokenizer.decode(encoded)

print(f"Original: {text}")
print(f"Encoded: {encoded}")
print(f"Decoded: {decoded}")

Original: The quick brown fox's jump.
Encoded: [464, 2068, 7586, 21831, 338, 4391, 13]
Decoded: The quick brown fox's jump.


In [44]:
tokenizer.encode("and established himself in a  villa ")

[392, 4920, 2241, 287, 257, 220, 4489, 64, 220]

In [45]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [46]:
len(vocab.items())

31596

In [47]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('”—whenever', 31591)
('•', 31592)
('\ufeffThe', 31593)
('<|endoftext|>', 31594)
('<|unk|>', 31595)


In [48]:
sen1 = "LLM from scratch"
sen2 = "The weather is cool"
text = sen1 + ' <|endoftext|> ' +sen2
text


'LLM from scratch <|endoftext|> The weather is cool'

In [49]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [50]:
tokenizer = SimpleTokenizerV2(vocab)

tokenizer.encode("and established himself in a huge villa hello blabla")

[5847, 12203, 15193, 15872, 5132, 15483, 28330, 31595, 31595]

In [51]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [52]:
tokenizer.encode(text)

[31595,
 28,
 11229,
 29553,
 17422,
 26216,
 265,
 31594,
 2396,
 26401,
 31595,
 26327,
 19459,
 26401,
 19982,
 32]

In [53]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the <|unk|> terraces of the palace.'

In [54]:
text = "abcabcad"
#a, b -> "ab"-> z -> 257
#"zczcad"
#z, c -> "zc" -> 258

In [55]:
# pip install tiktoken

In [56]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.12.0


In [57]:
tokenizer = tiktoken.get_encoding("gpt2")

In [58]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [59]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [60]:
# تأكد أن اسم الملف يطابق تماماً الملف الذي رفعته أو أنشأته
with open("pretraining_data.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# الآن قم بعمل encode للنص بالكامل
enc_text = tokenizer.encode(raw_text)

# احسب عدد الـ Tokens الكلي
print(len(enc_text))

765992


In [61]:
raw_text

Output hidden; open in https://colab.research.google.com to view.

In [62]:
enc_sample = enc_text[50:]
enc_sample

[198,
 10919,
 15485,
 13,
 921,
 743,
 4866,
 340,
 11,
 1577,
 340,
 1497,
 393,
 302,
 12,
 1904,
 340,
 739,
 262,
 2846,
 198,
 1659,
 262,
 4935,
 20336,
 13789,
 3017,
 351,
 428,
 46566,
 393,
 2691,
 379,
 198,
 2503,
 13,
 70,
 19028,
 13,
 2398,
 13,
 1002,
 345,
 389,
 407,
 5140,
 287,
 262,
 1578,
 1829,
 11,
 345,
 198,
 10594,
 423,
 284,
 2198,
 262,
 3657,
 286,
 262,
 1499,
 810,
 345,
 389,
 5140,
 878,
 198,
 3500,
 428,
 46566,
 13,
 198,
 198,
 19160,
 25,
 383,
 15640,
 286,
 25730,
 17628,
 198,
 198,
 13838,
 25,
 13514,
 31634,
 31233,
 198,
 198,
 26362,
 7536,
 25,
 3389,
 2808,
 11,
 6244,
 685,
 68,
 10482,
 1303,
 1433,
 5333,
 60,
 198,
 58,
 6943,
 2904,
 6153,
 25,
 3267,
 838,
 11,
 1160,
 1954,
 60,
 198,
 198,
 32065,
 25,
 3594,
 198,
 198,
 27275,
 900,
 21004,
 25,
 41002,
 12,
 23,
 198,
 198,
 11547,
 771,
 416,
 25,
 281,
 11614,
 4935,
 20336,
 13904,
 290,
 5264,
 6065,
 41913,
 198,
 198,
 8162,
 33303,
 3963,
 3336,
 21965,
 23680,
 402,


In [63]:
enc_sample[:5]

[198, 10919, 15485, 13, 921]

In [64]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [198, 10919, 15485, 13]
y:      [10919, 15485, 13, 921]


In [65]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[198] ----> 10919
[198, 10919] ----> 15485
[198, 10919, 15485] ----> 13
[198, 10919, 15485, 13] ----> 921


In [66]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))


 ----> what

what ----> soever

whatsoever ----> .

whatsoever. ---->  You


In [67]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.10.0+cu128


In [68]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [69]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [70]:
with open("pretraining_data.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [71]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[171, 119, 123, 464]]), tensor([[ 119,  123,  464, 4935]])]


In [72]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 119,  123,  464, 4935]]), tensor([[  123,   464,  4935, 20336]])]


In [73]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[  171,   119,   123,   464],
        [ 4935, 20336, 46566,   286],
        [  383, 15640,   286, 25730],
        [17628,    11,   198,  1525],
        [13514, 31634, 31233,   198],
        [  198,  1212, 46566,   318],
        [  329,   262,   779,   286],
        [ 2687,  6609,   287,   262]])

Targets:
 tensor([[  119,   123,   464,  4935],
        [20336, 46566,   286,   383],
        [15640,   286, 25730, 17628],
        [   11,   198,  1525, 13514],
        [31634, 31233,   198,   198],
        [ 1212, 46566,   318,   329],
        [  262,   779,   286,  2687],
        [ 6609,   287,   262,  1578]])


In [74]:
input_ids = torch.tensor([2, 3, 5, 1])

In [75]:
vocab_size = 6 # data
output_dim = 3 # hyper param ythat you can pick

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [76]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [77]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [78]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [79]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [80]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [81]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[  171,   119,   123,   464],
        [ 4935, 20336, 46566,   286],
        [  383, 15640,   286, 25730],
        [17628,    11,   198,  1525],
        [13514, 31634, 31233,   198],
        [  198,  1212, 46566,   318],
        [  329,   262,   779,   286],
        [ 2687,  6609,   287,   262]])

Inputs shape:
 torch.Size([8, 4])


In [82]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(token_embeddings)

torch.Size([8, 4, 256])


In [83]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# uncomment & execute the following line to see how the embedding layer weights look like
# print(pos_embedding_layer.weight)

In [84]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(pos_embeddings)

torch.Size([4, 256])


In [85]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(input_embeddings)

torch.Size([8, 4, 256])


Attention Mechanism

In [86]:
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.10.0+cu128


In [87]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [88]:
query = inputs[1]  # journey - > 2nd input token is the query
query

tensor([0.5500, 0.8700, 0.6600])

In [89]:
inputs.shape[0]

6

In [90]:
attn_scores_2 = torch.empty(6)

for i, key_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(key_i, query) # dot product (transpose not necessary here since they are 1-dim vectors)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [91]:
import math
attn_scores_2  = attn_scores_2 / math.sqrt(3)
attn_scores_2

tensor([0.5510, 0.8631, 0.8518, 0.4869, 0.4082, 0.6273])

In [92]:
res = 0.

for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]

print(res)
print(torch.dot(inputs[0], query))

tensor(0.9544)
tensor(0.9544)


In [93]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.)


In [94]:
# from numbers to prob - > sum to 1
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)

print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1515, 0.2070, 0.2046, 0.1421, 0.1313, 0.1635])
Sum: tensor(1.)


In [95]:
dummy_t = torch.tensor([[0.5510, 0.8631, 0.8518, 0.4869, 0.4082, 0.6273], [0.5510, 0.8631, 0.8518, 0.4869, 0.4082, 0.6273]])
dummy_t # dim = 0

tensor([[0.5510, 0.8631, 0.8518, 0.4869, 0.4082, 0.6273],
        [0.5510, 0.8631, 0.8518, 0.4869, 0.4082, 0.6273]])

In [96]:
dummy_t.T # dim = 1

tensor([[0.5510, 0.5510],
        [0.8631, 0.8631],
        [0.8518, 0.8518],
        [0.4869, 0.4869],
        [0.4082, 0.4082],
        [0.6273, 0.6273]])

In [97]:
attn_weights_2 = torch.softmax(attn_scores_2/math.sqrt(3), dim=0)

print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1583, 0.1896, 0.1883, 0.1526, 0.1458, 0.1654])
Sum: tensor(1.)


In [98]:
query = inputs[1]  # Jorney - > 2nd input token is the query
query

tensor([0.5500, 0.8700, 0.6600])

In [99]:
key = inputs[1]
key

tensor([0.5500, 0.8700, 0.6600])

In [100]:
value = inputs[1]
value

tensor([0.5500, 0.8700, 0.6600])

In [101]:
context_vec_2 = torch.zeros(3)
for i,value_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*value_i # weighted sum

print(context_vec_2)

tensor([0.4338, 0.6060, 0.5425])


In [102]:
attn_scores = torch.empty(6, 6)

for i, query_i in enumerate(inputs):
    for j, value_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(query_i, value_j)

print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [103]:
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [104]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [105]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 sum:", row_2_sum)

print("All row sums:", attn_weights.sum(dim=-1))

Row 2 sum: 1.0
All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [106]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [107]:
print("Previous 2nd context vector:", context_vec_2)

Previous 2nd context vector: tensor([0.4338, 0.6060, 0.5425])


In [108]:
x_2 = inputs[1] # second input element
d_in = inputs.shape[1] # the input embedding size, d=3
d_out = 2 # the output embedding size, d=2

In [109]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [110]:
W_query

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])

In [111]:
W_key

Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]])

In [112]:
W_value

Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]])

In [113]:
x_2

tensor([0.5500, 0.8700, 0.6600])

In [114]:
query_2 = x_2 @ W_query # _2 because it's with respect to the 2nd input element
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

print(query_2)

tensor([0.4306, 1.4551])


In [115]:
keys = inputs @ W_key
values = inputs @ W_value

print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


In [116]:
keys_2 = keys[1] # Python starts index at 0
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)


In [117]:
attn_scores_2 = query_2 @ keys.T # All attention scores for given query
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [118]:
d_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [119]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


In [120]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [121]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


In [122]:
# Reuse the query and key weight matrices of the
# SelfAttention_v2 object from the previous section for convenience
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [123]:
context_length = attn_scores.shape[0]
context_length

6

In [124]:
 torch.tril(torch.ones(context_length, context_length))

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

In [125]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [126]:
attn_weights

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)

In [127]:
masked_simple = attn_weights*mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


In [128]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


In [129]:
torch.triu(torch.ones(context_length, context_length), diagonal=1)

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])

In [130]:
# 1. أولاً: نحدد الحجم (T) بناءً على مصفوفة النتائج التي لديك
T = attn_scores.shape[-1]

# 2. ثانياً: ننشئ القناع (mask)
mask = torch.triu(torch.ones(T, T), diagonal=1)

# 3. ثالثاً: نطبق القناع (السطر الذي كان يعطيك خطأ سابقاً)
masked_attn_scores = attn_scores.masked_fill(mask.bool(), -torch.inf)

# 4. أخيراً: نعرض النتيجة
print(masked_attn_scores)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [131]:
import torch

# 1. لنفترض أن لدينا 6 كلمات (tokens) وكل كلمة لها متجه طوله 3
inputs = torch.tensor(
  [[0.43, 0.15, 0.89],
   [0.55, 0.87, 0.66],
   [0.57, 0.85, 0.64],
   [0.22, 0.58, 0.33],
   [0.77, 0.25, 0.50],
   [0.19, 0.69, 0.56]]
)

# 2. حساب attn_scores (ببساطة عن طريق ضرب المصفوفة في نفسها)
# هذا السطر هو الذي كان ينقصك
attn_scores = inputs @ inputs.T

# 3. الآن يمكنك حساب طول السياق وإنشاء القناع
context_length = attn_scores.shape[-1]
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)

# 4. تطبيق الـ Mask
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)

print(masked)

tensor([[0.9995,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.9544, 1.4950,   -inf,   -inf,   -inf,   -inf],
        [0.9422, 1.4754, 1.4570,   -inf,   -inf,   -inf],
        [0.4753, 0.8434, 0.8296, 0.4937,   -inf,   -inf],
        [0.8136, 0.9710, 0.9714, 0.4794, 0.9054,   -inf],
        [0.6836, 1.0744, 1.0532, 0.6268, 0.5988, 0.8258]])


In [132]:
import torch

# نفترض أن d_k (أبعاد الـ key) هي 3
keys = torch.tensor([[0.2, 0.4, 0.6],
                     [0.1, 0.5, 0.8],
                     [0.3, 0.3, 0.3],
                     [0.9, 0.1, 0.1],
                     [0.5, 0.5, 0.5],
                     [0.2, 0.7, 0.1]])

# الآن سيعمل السطر الخاص بك بنجاح
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4226, 0.5774, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2698, 0.3670, 0.3632, 0.0000, 0.0000, 0.0000],
        [0.2235, 0.2764, 0.2742, 0.2259, 0.0000, 0.0000],
        [0.1973, 0.2160, 0.2161, 0.1626, 0.2080, 0.0000],
        [0.1539, 0.1929, 0.1905, 0.1490, 0.1466, 0.1671]])


In [133]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5) # dropout rate of 50%
example = torch.ones(6, 6) # create a matrix of ones

print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [134]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5396, 0.7341, 0.7263, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5528, 0.5484, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4320, 0.0000, 0.3253, 0.0000, 0.0000],
        [0.0000, 0.3858, 0.3811, 0.2979, 0.2931, 0.0000]])


In [135]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) # 2 inputs with 6 tokens each, and each token has embedding dimension 3

torch.Size([2, 6, 3])


In [136]:
import torch
import torch.nn as nn  # هذا هو السطر الذي يحل مشكلة 'nn'

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # استخدام register_buffer يضمن بقاء الـ mask مع النموذج وفي الـ GPU
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # حساب التقارب مع مراعاة أبعاد الـ Batch
        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

# --- لتشغيل الاختبار بالأسفل، نحتاج لتعريف المتغيرات الناقصة ---
torch.manual_seed(123)

# قيم افتراضية للتجربة (تأكد أنها مطابقة لما لديك)
d_in, d_out = 3, 2
batch = torch.ones((2, 4, d_in)) # Batch size=2, Context=4, Embedding=3

context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)

context_vecs = ca(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-1.0523, -0.1759],
         [-1.0523, -0.1759],
         [-1.0523, -0.1759],
         [-1.0523, -0.1759]],

        [[-1.0523, -0.1759],
         [-1.0523, -0.1759],
         [-1.0523, -0.1759],
         [-1.0523, -0.1759]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 4, 2])


In [137]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


torch.manual_seed(123)

context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-1.0523, -0.1759,  1.0201,  0.6414],
         [-1.0523, -0.1759,  1.0201,  0.6414],
         [-1.0523, -0.1759,  1.0201,  0.6414],
         [-1.0523, -0.1759,  1.0201,  0.6414]],

        [[-1.0523, -0.1759,  1.0201,  0.6414],
         [-1.0523, -0.1759,  1.0201,  0.6414],
         [-1.0523, -0.1759,  1.0201,  0.6414],
         [-1.0523, -0.1759,  1.0201,  0.6414]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 4, 4])


In [138]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`,
        # this will result in errors in the mask creation further below.
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3289, 0.1332],
         [0.3289, 0.1332],
         [0.3289, 0.1332],
         [0.3289, 0.1332]],

        [[0.3289, 0.1332],
         [0.3289, 0.1332],
         [0.3289, 0.1332],
         [0.3289, 0.1332]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 4, 2])


In [139]:
# (b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

print(a @ a.transpose(2, 3))

tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


In [140]:
first_head = a[0, 0, :, :]
first_res = first_head @ first_head.T
print("First head:\n", first_res)

second_head = a[0, 1, :, :]
second_res = second_head @ second_head.T
print("\nSecond head:\n", second_res)

First head:
 tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

Second head:
 tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


Implementing a GPT model from Scratch To Generate Text

In [141]:
from importlib.metadata import version

print("matplotlib version:", version("matplotlib"))
print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

matplotlib version: 3.10.0
torch version: 2.10.0+cu128
tiktoken version: 0.12.0


In [142]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers (Decoder layers)
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

In [143]:
import torch
import torch.nn as nn


class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"]) # overfitting

        # Use a placeholder for TransformerBlock
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(12)])

        # Use a placeholder for LayerNorm
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x) # 5, 8, 201, ...
        #prob = torch.softmax(logits) # 0.01, 0.03, 0.2, ... propapilities
        return logits


class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # A simple placeholder

    def forward(self, x):
        # This block does nothing and just returns its input.
        return x


class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        # The parameters here are just to mimic the LayerNorm interface.

    def forward(self, x):
        # This layer does nothing and just returns its input.
        return x

In [144]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [145]:
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)

logits = model(batch)
print("Output shape:", logits.shape)
print(logits)

Output shape: torch.Size([2, 4, 50257])
tensor([[[-1.2034,  0.3201, -0.7130,  ..., -1.5548, -0.2390, -0.4667],
         [-0.1192,  0.4539, -0.4432,  ...,  0.2392,  1.3469,  1.2430],
         [ 0.5307,  1.6720, -0.4695,  ...,  1.1966,  0.0111,  0.5835],
         [ 0.0139,  1.6755, -0.3388,  ...,  1.1586, -0.0435, -1.0400]],

        [[-1.0908,  0.1798, -0.9484,  ..., -1.6047,  0.2439, -0.4530],
         [-0.7860,  0.5581, -0.0610,  ...,  0.4835, -0.0077,  1.6621],
         [ 0.3567,  1.2698, -0.6398,  ..., -0.0162, -0.1296,  0.3717],
         [-0.2407, -0.7349, -0.5102,  ...,  2.0057, -0.3694,  0.1814]]],
       grad_fn=<UnsafeViewBackward0>)


In [146]:
torch.manual_seed(123)

# create 2 training examples with 5 dimensions (features) each
batch_example = torch.randn(2, 5)

layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
print(out)

tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>)


In [147]:
mean = out.mean(dim=-1, keepdim=True)
var = out.var(dim=-1, keepdim=True)

print("Mean:\n", mean)
print("Variance:\n", var)

Mean:
 tensor([[0.1324],
        [0.2170]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[0.0231],
        [0.0398]], grad_fn=<VarBackward0>)


In [148]:
out_norm = (out - mean) / torch.sqrt(var)
print("Normalized layer outputs:\n", out_norm)

mean = out_norm.mean(dim=-1, keepdim=True)
var = out_norm.var(dim=-1, keepdim=True)
print("Mean:\n", mean)
print("Variance:\n", var)

Normalized layer outputs:
 tensor([[ 0.6159,  1.4126, -0.8719,  0.5872, -0.8719, -0.8719],
        [-0.0189,  0.1121, -1.0876,  1.5173,  0.5647, -1.0876]],
       grad_fn=<DivBackward0>)
Mean:
 tensor([[9.9341e-09],
        [1.9868e-08]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


In [149]:
torch.set_printoptions(sci_mode=False)
print("Mean:\n", mean)
print("Variance:\n", var)

Mean:
 tensor([[0.0000],
        [0.0000]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


In [150]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [151]:
ln = LayerNorm(emb_dim=6)
out_ln = ln(out)

In [152]:
mean = out_ln.mean(dim=-1, keepdim=True)
var = out_ln.var(dim=-1, unbiased=False, keepdim=True)

print("Mean:\n", mean)
print("Variance:\n", var)

Mean:
 tensor([[ 0.0000],
        [-0.0000]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[0.9995],
        [0.9997]], grad_fn=<VarBackward0>)


In [153]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

In [154]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

In [155]:
print(GPT_CONFIG_124M["emb_dim"])

768


In [156]:
ffn = FeedForward(GPT_CONFIG_124M)

# input shape: [batch_size, num_token, emb_size]
x = torch.rand(2, 3, 768)
out = ffn(x)
print(out.shape)

torch.Size([2, 3, 768])


In [157]:
class ExampleDeepNeuralNetwork(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), GELU())
        ])

    def forward(self, x):
        for layer in self.layers:
            # Compute the output of the current layer
            layer_output = layer(x)
            # Check if shortcut can be applied
            if self.use_shortcut and x.shape == layer_output.shape:
                x = x + layer_output
            else:
                x = layer_output
        return x


def print_gradients(model, x):
    # Forward pass
    output = model(x)
    target = torch.tensor([[0.]])

    # Calculate loss based on how close the target
    # and output are
    loss = nn.MSELoss()
    loss = loss(output, target)

    # Backward pass to calculate the gradients
    loss.backward()

    for name, param in model.named_parameters():
        if 'weight' in name:
            # Print the mean absolute gradient of the weights
            print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")

In [158]:
layer_sizes = [3, 3, 3, 3, 3, 1]

sample_input = torch.tensor([[1., 0., -1.]])

torch.manual_seed(123)
model_without_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=False
)
print_gradients(model_without_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.00020173584925942123
layers.1.0.weight has gradient mean of 0.00012011159560643137
layers.2.0.weight has gradient mean of 0.0007152040489017963
layers.3.0.weight has gradient mean of 0.0013988736318424344
layers.4.0.weight has gradient mean of 0.005049645435065031


In [159]:
torch.manual_seed(123)
model_with_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=True
)
print_gradients(model_with_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.22169791162014008
layers.1.0.weight has gradient mean of 0.20694105327129364
layers.2.0.weight has gradient mean of 0.32896995544433594
layers.3.0.weight has gradient mean of 0.2665732204914093
layers.4.0.weight has gradient mean of 1.3258540630340576


In [160]:
import torch
import torch.nn as nn

# ملاحظة: تأكد أنك قمت بتشغيل خلية كلاس MultiHeadAttention و LayerNorm و FeedForward أولاً
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x

In [161]:
torch.manual_seed(123)

x = torch.rand(2, 4, 768)  # Shape: [batch_size, num_tokens, emb_dim]
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])


In [162]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(12)]) # small, m24, Large36

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x) # vocab size
        # softmax -> prob
        return logits

In [163]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape: torch.Size([2, 4, 50257])
tensor([[[ 0.3613,  0.4222, -0.0711,  ...,  0.3483,  0.4661, -0.2838],
         [-0.1792, -0.5660, -0.9485,  ...,  0.0477,  0.5181, -0.3168],
         [ 0.7120,  0.0332,  0.1085,  ...,  0.1018, -0.4327, -0.2553],
         [-1.0076,  0.3418, -0.1190,  ...,  0.7195,  0.4023,  0.0532]],

        [[-0.2564,  0.0900,  0.0335,  ...,  0.2659,  0.4454, -0.6806],
         [ 0.1230,  0.3653, -0.2074,  ...,  0.7705,  0.2710,  0.2246],
         [ 1.0558,  1.0318, -0.2800,  ...,  0.6936,  0.3205, -0.3178],
         [-0.1565,  0.3926,  0.3288,  ...,  1.2630, -0.1858,  0.0388]]],
       grad_fn=<UnsafeViewBackward0>)


In [164]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

Total number of parameters: 163,009,536


In [165]:
print("Token embedding layer shape:", model.tok_emb.weight.shape)
print("Output layer shape:", model.out_head.weight.shape)

Token embedding layer shape: torch.Size([50257, 768])
Output layer shape: torch.Size([50257, 768])


In [166]:
total_params_gpt2 =  total_params - sum(p.numel() for p in model.out_head.parameters())
print(f"Number of trainable parameters considering weight tying: {total_params_gpt2:,}")

Number of trainable parameters considering weight tying: 124,412,160


In [ ]:
# Calculate the total size in bytes (assuming float32, 4 bytes per parameter)
total_size_bytes = total_params * 4

# Convert to megabytes
total_size_mb = total_size_bytes / (1024 * 1024)

print(f"Total size of the model: {total_size_mb:.2f} MB")

Total size of the model: 621.83 MB


In [168]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context
    for _ in range(max_new_tokens):
    #while token != '<|endoftext|>':

        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]

        # Get the predictions
        with torch.no_grad(): # model.infrance_mode()
            logits = model(idx_cond)

        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

In [169]:
start_context = "Hello, I am"

encoded = tokenizer.encode(start_context)
print("encoded:", encoded)

encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor.shape:", encoded_tensor.shape)

encoded: [15496, 11, 314, 716]
encoded_tensor.shape: torch.Size([1, 4])


In [170]:
model.eval() # disable dropout

out = generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=6,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output:", out)
print("Output length:", len(out[0]))

RuntimeError: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)

In [171]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

TypeError: argument 'tokens': 'list' object cannot be interpreted as an integer

m5

In [ ]:
from importlib.metadata import version

pkgs = ["matplotlib",
        "numpy",
        "tiktoken",
        "torch",
        "tensorflow" # For OpenAI's pretrained weights
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

matplotlib version: 3.10.0
numpy version: 2.0.2
tiktoken version: 0.12.0
torch version: 2.10.0+cu128
tensorflow version: 2.19.0


In [ ]:
import torch
# If the `previous_chapters.py` file is not available locally,
# you can import it from the `llms-from-scratch` PyPI package.
# For details, see: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# E.g.,
# from llms_from_scratch.ch04 import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();  # Disable dropout during inference

In [ ]:
import tiktoken

# Alternatively:
# from llms_from_scratch.ch04 import generate_text_simple

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

# هذا هو الكود الصحيح للاستدعاء بناءً على تعريفك:
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,  # يجب أن يكون الاسم مطابقاً تماماً لما كتبته في def
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren


In [ ]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]]) #  " really like chocolate"]

In [ ]:
with torch.no_grad():
    logits = model(inputs)

probas = torch.softmax(logits, dim=-1) # Probability of each token in vocabulary
print(probas.shape) # Shape: (batch_size, num_tokens, vocab_size)

torch.Size([2, 3, 50257])


In [ ]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print("Token IDs:\n", token_ids)

Token IDs:
 tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])


In [ ]:
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

Targets batch 1:  effort moves you
Outputs batch 1:  Armed heNetflix


In [ ]:
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 1:", target_probas_1)

text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 2:", target_probas_2)

Text 1: tensor([0.0001, 0.0000, 0.0000])
Text 2: tensor([0.0000, 0.0001, 0.0000])


In [ ]:
# Compute logarithm of all token probabilities
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print(log_probas)

tensor([ -9.5042, -10.3796, -11.3677, -11.4798,  -9.7764, -12.2561])


In [ ]:
# Calculate the average probability for each token
avg_log_probas = torch.mean(log_probas)
print(avg_log_probas)

tensor(-10.7940)


In [ ]:
neg_avg_log_probas = avg_log_probas * -1
print(neg_avg_log_probas)

tensor(10.7940)


In [ ]:
# Logits have shape (batch_size, num_tokens, vocab_size)
print("Logits shape:", logits.shape)

# Targets have shape (batch_size, num_tokens)
print("Targets shape:", targets.shape)

Logits shape: torch.Size([2, 3, 50257])
Targets shape: torch.Size([2, 3])


In [ ]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()

print("Flattened logits:", logits_flat.shape)
print("Flattened targets:", targets_flat.shape)

Flattened logits: torch.Size([6, 50257])
Flattened targets: torch.Size([6])


In [ ]:
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

tensor(10.7940)


In [ ]:
perplexity = torch.exp(loss)
print(perplexity)

tensor(48725.8203)


In [ ]:
import os  # هذا السطر هو الحل!
file_path = "pretraining_data.txt"

# بما أن الملف موجود عندك، لا داعي للـ URL ولا للـ requests
if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()
    print(f"تم تحميل الملف بنجاح! عدد الحروف: {len(text_data)}")
else:
    print(f"خطأ: الملف {file_path} غير موجود. تأكد من رفعه إلى مساحة عمل Colab.")

تم تحميل الملف بنجاح! عدد الحروف: 2795192


In [ ]:
# First 99 characters
print(text_data[:99])

﻿The Project Gutenberg eBook of The Adventures of Sherlock Holmes,
by Arthur Conan Doyle

This eBoo


In [ ]:
# Last 99 characters
print(text_data[-99:])

, I saw no shadow of another parting
from her.


*** END OF THE PROJECT GUTENBERG EBOOK 1400 ***





In [ ]:
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))

print("Characters:", total_characters)
print("Tokens:", total_tokens)

Characters: 2795192
Tokens: 765992


In [ ]:
# Alternatively:
# from llms_from_scratch.ch02 import create_dataloader_v1

# Train/validation ratio
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [ ]:
# Sanity check

if total_tokens * (train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the training loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "increase the `training_ratio`")

if total_tokens * (1-train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the validation loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "decrease the `training_ratio`")

In [ ]:
print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\nValidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

Train loader:
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256])

In [ ]:
train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()

print("Training tokens:", train_tokens)
print("Validation tokens:", val_tokens)
print("All tokens:", train_tokens + val_tokens)

Training tokens: 688128
Validation tokens: 77568
All tokens: 765696


In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

+++++_

In [ ]:
# 1. تحديد الجهاز (سيكتشف الـ GPU تلقائياً)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # التحقق من إصدار تورش لـ MPS
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print(f"Using {device} device.")

# 2. نقل النموذج للجهاز (تعديل: إضافة تعيين المتغير لضمان العمل مع كل أنواع الموديلات)
model = model.to(device)

# 3. حساب الخسارة (Loss)
torch.manual_seed(123)

with torch.no_grad(): # تعطيل حساب المشتقات للتقييم فقط
    # هنا نتأكد من تمرير الـ device للدالة لكي تقوم بالنقل الداخلي
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Using cuda device.


KeyboardInterrupt: 

In [ ]:
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    # Initialize lists to track losses and tokens seen
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    # Main training loop
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # Reset loss gradients from previous batch iteration
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() # Calculate loss gradients
            optimizer.step() # Update model weights using loss gradients
            tokens_seen += input_batch.numel()
            global_step += 1

            # Optional evaluation step
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # Print a sample text after each epoch
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # Compact print format
    model.train()

In [ ]:
# حفظ النموذج الحالي
torch.save(model.state_dict(), "model_step_4500.pth")
print("Model saved successfully!")

In [ ]:
# Note:
# Uncomment the following code to calculate the execution time
# import time
# start_time = time.time()

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

num_epochs = 5
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=500, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)

# Note:
# Uncomment the following code to show the execution time
# end_time = time.time()
# execution_time_minutes = (end_time - start_time) / 60
# print(f"Training completed in {execution_time_minutes:.2f} minutes.")

هنا دربت المودل(هنا الي وصلت له)

In [ ]:
# حفظ النموذج الحالي
torch.save(model.state_dict(), "model_step_4500.pth")
print("Model saved successfully!")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # Plot training and validation loss against epochs
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))  # only show integer labels on x-axis

    # Create a second x-axis for tokens seen
    ax2 = ax1.twiny()  # Create a second x-axis that shares the same y-axis
    ax2.plot(tokens_seen, train_losses, alpha=0)  # Invisible plot for aligning ticks
    ax2.set_xlabel("Tokens seen")

    fig.tight_layout()  # Adjust layout to make room
    plt.savefig("loss-plot.pdf")
    plt.show()

epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

In [ ]:
# NEW: use CPU here as inference is cheap with
# this model and to ensure readers get same results in the
# remaining sections of this book
inference_device = torch.device("cpu")

model.to(inference_device)
model.eval()

tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(inference_device),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
vocab = {
    "closer": 0,
    "every": 1,
    "effort": 2,
    "forward": 3,
    "inches": 4,
    "moves": 5,
    "pizza": 6,
    "toward": 7,
    "you": 8,
}

inverse_vocab = {v: k for k, v in vocab.items()}

# Suppose input is "every effort moves you", and the LLM
# returns the following logits for the next token:
next_token_logits = torch.tensor(
    [4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79]
)

probas = torch.softmax(next_token_logits, dim=0)
next_token_id = torch.argmax(probas).item()

# The next generated token is then as follows:
print(inverse_vocab[next_token_id])

In [ ]:
torch.manual_seed(123)
next_token_id = torch.multinomial(probas, num_samples=1).item()
print(inverse_vocab[next_token_id])

In [ ]:
def print_sampled_tokens(probas):
    torch.manual_seed(123) # Manual seed for reproducibility
    sample = [torch.multinomial(probas, num_samples=1).item() for i in range(1_000)]
    sampled_ids = torch.bincount(torch.tensor(sample), minlength=len(probas))
    for i, freq in enumerate(sampled_ids):
        print(f"{freq} x {inverse_vocab[i]}")

print_sampled_tokens(probas)

In [ ]:
def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)

# Temperature values
temperatures = [1, 0.1, 5]  # Original, higher confidence, and lower confidence

# Calculate scaled probabilities
scaled_probas = [softmax_with_temperature(next_token_logits, T) for T in temperatures]

In [ ]:
# Plotting
x = torch.arange(len(vocab))
bar_width = 0.15

fig, ax = plt.subplots(figsize=(5, 3))
for i, T in enumerate(temperatures):
    rects = ax.bar(x + i * bar_width, scaled_probas[i], bar_width, label=f'Temperature = {T}')

ax.set_ylabel('Probability')
ax.set_xticks(x)
ax.set_xticklabels(vocab.keys(), rotation=90)
ax.legend()

plt.tight_layout()
plt.savefig("temperature-plot.pdf")
plt.show()

In [ ]:
print_sampled_tokens(scaled_probas[1])

In [ ]:
print_sampled_tokens(scaled_probas[2])

In [ ]:
top_k = 3
top_logits, top_pos = torch.topk(next_token_logits, top_k)

print("Top logits:", top_logits)
print("Top positions:", top_pos)

In [ ]:
new_logits = torch.where(
    condition=next_token_logits < top_logits[-1],
    input=torch.tensor(float("-inf")),
    other=next_token_logits
)

print(new_logits)

In [ ]:
topk_probas = torch.softmax(new_logits, dim=0)
print(topk_probas)

In [ ]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):

    # For-loop is the same as before: Get logits, and only focus on last time step
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # New: Filter logits with top_k sampling
        if top_k is not None:
            # Keep only top_k values
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        # New: Apply temperature scaling
        if temperature > 0.0:
            logits = logits / temperature

            # New (not in book): numerical stability tip to get equivalent results on mps device
            # subtract rowwise max before softmax
            logits = logits - logits.max(dim=-1, keepdim=True).values

            # Apply softmax to get probabilities
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

        # Otherwise same as before: get idx of the vocab entry with the highest logits value
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)

        if idx_next == eos_id:  # Stop generating early if end-of-sequence token is encountered and eos_id is specified
            break

        # Same as before: append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch_size, num_tokens+1)

    return idx

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(inference_device),
    max_new_tokens=15,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=25,
    temperature=1.4
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
torch.save(model.state_dict(), "model.pth")

In [ ]:
model = GPTModel(GPT_CONFIG_124M)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Use PyTorch 2.9 or newer for stable mps results
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)

model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))
model.eval();

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    },
    "model_and_optimizer.pth"
)

In [ ]:
checkpoint = torch.load("model_and_optimizer.pth", weights_only=True)

model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train();

++++++_

In [1]:
import json
import torch
import torch.nn.functional as F

english_data = [
    {"input": "i is going to school", "output": "i am going to school"},
    {"input": "she go to the store", "output": "she goes to the store"},
    {"input": "he have a car", "output": "he has a car"},
    {"input": "they was happy", "output": "they were happy"}
]

with open('en_data.json', 'w') as f:
    json.dump(english_data, f)

هنا بعد ما ملف المودل حقي ما اشتغل جربت ذي الطريقه

In [5]:
import torch
import torch.nn as nn
import json

# 1. تعريف الإعدادات (نفس الإعدادات السابقة)
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

# 2. تعريف القالب (GPTModel) - (انسخه كما هو)
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[nn.TransformerEncoderLayer(d_model=cfg["emb_dim"], nhead=cfg["n_heads"], batch_first=True) for _ in range(cfg["n_layers"])])
        self.final_norm = nn.LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = self.drop_emb(tok_embeds + pos_embeds)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)

# 3. بناء موديل جديد (بدون تحميل ملف تالف)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
model.train()

# 4. تدريب سريع للتأكد أن الموديل يعمل
print("تم بناء نموذج جديد وجاري التدريب...")
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
# (استخدمنا بيانات بسيطة مدمجة هنا)
input_data = torch.randint(0, 50257, (1, 10)).to(device)
logits = model(input_data)
loss = logits.sum()
loss.backward()
optimizer.step()

print("تم بنجاح! الموديل يعمل الآن في الذاكرة.")

تم بناء نموذج جديد وجاري التدريب...
تم بنجاح! الموديل يعمل الآن في الذاكرة.


In [8]:
import torch
import torch.nn.functional as F
from transformers import GPT2Tokenizer

# 1. إعداد المحلل (Tokenizer)
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# 2. بيانات التدريب الإنجليزية
english_data = [
    {"input": "i is going to school", "output": "i am going to school"},
    {"input": "she go to the store", "output": "she goes to the store"},
    {"input": "he have a car", "output": "he has a car"}
]

# 3. التدريب الفعلي
print("جاري التدريب على الجمل الإنجليزية...")
model.train() # نستخدم الموديل الذي أنشأناه في الخلية السابقة
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

for epoch in range(50):
    for entry in english_data:
        optimizer.zero_grad()

        # تحويل النص لأرقام
        input_ids = torch.tensor(tokenizer.encode(entry['input'])).unsqueeze(0).to(device)
        target_ids = torch.tensor(tokenizer.encode(entry['output'])).unsqueeze(0).to(device)

        # التمرير
        logits = model(input_ids)

        # حساب الخطأ (Loss)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), target_ids.view(-1))

        loss.backward()
        optimizer.step()

print("التدريب اكتمل! لنختبر الموديل الآن.")

# 4. الاختبار (توليد التصحيح)
model.eval()
test_prompt = "she go to the store"
input_ids = torch.tensor(tokenizer.encode(test_prompt)).unsqueeze(0).to(device)

with torch.no_grad():
    # نولد 5 كلمات إضافية كحد أقصى
    generated_ids = input_ids
    for _ in range(5):
        logits = model(generated_ids)
        next_token = torch.argmax(logits[:, -1, :], dim=-1).unsqueeze(0)
        generated_ids = torch.cat((generated_ids, next_token), dim=1)

result = tokenizer.decode(generated_ids[0].tolist())
print("-" * 30)
print(f"الجملة الأصلية: {test_prompt}")
print(f"التصحيح الناتج: {result}")

جاري التدريب على الجمل الإنجليزية...
التدريب اكتمل! لنختبر الموديل الآن.
------------------------------
الجملة الأصلية: she go to the store
التصحيح الناتج: she go to the store store store store store store


In [9]:
import torch.nn.functional as F

model.eval()
test_prompt = "she go to the store"
input_ids = torch.tensor(tokenizer.encode(test_prompt)).unsqueeze(0).to(device)

with torch.no_grad():
    generated_ids = input_ids
    for _ in range(5):
        logits = model(generated_ids)
        # 1. نأخذ التوقعات الأخيرة
        next_token_logits = logits[:, -1, :] / 0.7  # درجة الحرارة (Temperature)

        # 2. نمنع النموذج من اختيار نفس الكلمة فوراً (بمعاقبة الاحتمالية)
        next_token_logits[:, generated_ids[0, -1]] = -float('Inf')

        # 3. نختار بشكل عشوائي من بين أفضل الكلمات (Sampling)
        probs = F.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        generated_ids = torch.cat((generated_ids, next_token), dim=1)

result = tokenizer.decode(generated_ids[0].tolist())
print("-" * 30)
print(f"التصحيح الجديد: {result}")


------------------------------
التصحيح الجديد: she go to the store averaged store membership the displays


In [12]:
import json
import torch
import torch.nn.functional as F

# 1. تعريف البيانات الصحيحة برمجياً (لتجنب مشاكل الملف التالف)
data = [
    {"input": "i is happy", "output": "i am happy"},
    {"input": "she go to school", "output": "she goes to school"},
    {"input": "he have a dog", "output": "he has a dog"},
    {"input": "they was late", "output": "they were late"},
    {"input": "we has work", "output": "we have work"},
    {"input": "it do look nice", "output": "it does look nice"},
    {"input": "he play football", "output": "he plays football"},
    {"input": "she sing well", "output": "she sings well"},
    {"input": "they runs fast", "output": "they run fast"},
    {"input": "he drink milk", "output": "he drinks milk"},
    {"input": "she walk home", "output": "she walks home"},
    {"input": "he eat breakfast", "output": "he eats breakfast"}
] * 10

# 2. حفظ البيانات في الملف (لضمان التنسيق الصحيح)
with open('en_data.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print("تم إصلاح ملف البيانات بنجاح!")

# 3. التدريب
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

print("جاري التدريب...")
for epoch in range(50):
    for entry in data:
        optimizer.zero_grad()
        input_ids = torch.tensor(tokenizer.encode(entry['input'])).unsqueeze(0).to(device)
        target_ids = torch.tensor(tokenizer.encode(entry['output'])).unsqueeze(0).to(device)

        logits = model(input_ids)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), target_ids.view(-1))

        loss.backward()
        optimizer.step()

print("اكتمل التدريب!")

تم إصلاح ملف البيانات بنجاح!
جاري التدريب...
اكتمل التدريب!


In [16]:
import torch

def generate_text(model, idx, max_new_tokens):
    # نكرر التوقع لعدد معين من الكلمات
    for _ in range(max_new_tokens):
        # الحصول على توقعات النموذج
        logits = model(idx)
        # نأخذ التوقعات للكلمة الأخيرة فقط
        logits = logits[:, -1, :]
        # الحصول على الكلمة ذات الاحتمالية الأعلى
        _, next_idx = torch.topk(logits, k=1, dim=-1)
        # إضافة الكلمة الجديدة للمدخلات
        idx = torch.cat((idx, next_idx), dim=1)
    return idx

# الاختبار باستخدام الدالة الجديدة
model.eval()
test_prompts = ["he run fast", "she need help", "they is ready"]

print("--- اختبار ذكاء النموذج (يدوي) ---")
for prompt in test_prompts:
    input_ids = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0).to(device)

    with torch.no_grad():
        output_ids = generate_text(model, input_ids, max_new_tokens=3)

    result = tokenizer.decode(output_ids[0].tolist())
    print(f"الخاطئ: {prompt}  -->  التصحيح الناتج: {result}")

--- اختبار ذكاء النموذج (يدوي) ---
الخاطئ: he run fast  -->  التصحيح الناتج: he run fast breakfast breakfastthey
الخاطئ: she need help  -->  التصحيح الناتج: she need help well well well
الخاطئ: they is ready  -->  التصحيح الناتج: they is ready fast fast fast
